# Credit Risk Early Warning System
## Notebook 02 — Feature Engineering & Selection

**Objective:**  
Decide, with evidence, which engineered features earn a place in the final model.

**Core Rule:**  
A feature survives only if it satisfies at least **two of three** criteria:
1. Strong business rationale
2. Measurable signal (distribution separation, correlation, or model importance)
3. Easy to explain in a 30-second interview answer

We are not optimizing for leaderboard performance. We are optimizing for a defensible, interpretable, recruiter-credible feature set.

**Output of this notebook:**  
A locked `FINAL_MODEL_FEATURES` list, written to `src/features.py`, that becomes the single source of truth for modeling, calibration, and the Streamlit app.

---
## Section 0 — Setup

In [ ]:
import sys
import os
import glob
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

sys.path.append(os.path.join(os.getcwd(), '..', 'src'))

from preprocessing import prepare_data
from features import create_all_features

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
})

COLOR_DEFAULT    = '#c0392b'
COLOR_NONDEFAULT = '#2980b9'
COLOR_NEUTRAL    = '#7f8c8d'

print('Setup complete.')

In [ ]:
def load_lending_club_data(search_dirs=None):
    """Locate and load the LendingClub CSV (plain or gzip-compressed)."""
    if search_dirs is None:
        base = os.path.join(os.getcwd(), '..')
        search_dirs = [os.path.join(base, 'datasets'), os.path.join(base, 'data'), base]

    for directory in search_dirs:
        for pattern in ['*.csv', '*.csv.gz']:
            matches = glob.glob(os.path.join(directory, pattern))
            if matches:
                path = matches[0]
                print(f'Loading: {path}')
                return pd.read_csv(path, low_memory=False)

    raise FileNotFoundError('No CSV or CSV.GZ file found in datasets/ or data/.')


output_dir = os.path.join(os.getcwd(), '..', 'outputs')

raw_df = load_lending_club_data()
clean_df = prepare_data(raw_df.copy(), output_dir=output_dir)
df = create_all_features(clean_df.copy(), output_dir=output_dir)

print(f'\nWorking dataset: {df.shape[0]:,} loans x {df.shape[1]} columns')

---
## Section 1 — Evaluation Framework

For every candidate feature we compute three pieces of evidence:

1. **Distribution separation** — do defaulters and non-defaulters look different?
2. **Point-biserial correlation with target** — linear association strength
3. **Logistic Regression coefficient ranking** — does it carry weight once all other features are present?

The helper functions below compute these consistently for every feature reviewed in this notebook.

In [ ]:
def feature_evidence(df, feature, target='target'):
    """
    Return a one-row summary of evidence for a single feature:
    mean for non-defaulters, mean for defaulters, % difference, correlation.
    """
    mean_0 = df[df[target] == 0][feature].mean()
    mean_1 = df[df[target] == 1][feature].mean()
    pct_diff = ((mean_1 - mean_0) / abs(mean_0)) * 100 if mean_0 != 0 else np.nan
    corr = df[feature].corr(df[target])
    return {
        'feature': feature,
        'mean_non_default': round(mean_0, 4),
        'mean_default': round(mean_1, 4),
        'pct_difference': round(pct_diff, 1),
        'correlation_with_target': round(corr, 4),
    }


def plot_distribution(df, feature, title, ax=None):
    """Overlapping histograms of feature by target, clipped to 1st-99th percentile."""
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 4))
    for target_val, label, color in [(0, 'Fully Paid', COLOR_NONDEFAULT), (1, 'Charged Off', COLOR_DEFAULT)]:
        vals = df[df['target'] == target_val][feature].dropna()
        lo, hi = vals.quantile(0.01), vals.quantile(0.99)
        vals = vals.clip(lo, hi)
        ax.hist(vals, bins=50, alpha=0.5, color=color, label=label, density=True)
    ax.set_title(title, fontsize=11)
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)
    return ax


print('Evaluation helpers defined.')

---
## Section 2 — Feature-by-Feature Evaluation

Each feature below follows the same format:  
**Business Rationale → Distribution → Default Rate Relationship → Decision**

### 2.1 `monthly_payment_burden`

**Formula:** `installment / (annual_inc / 12)`

**Business Rationale:** What fraction of a borrower's monthly income goes directly to this loan's payment? This is the single most intuitive affordability check in consumer lending.

In [ ]:
plot_distribution(df, 'monthly_payment_burden', 'Monthly Payment Burden by Outcome')
plt.tight_layout()
plt.show()

pd.DataFrame([feature_evidence(df, 'monthly_payment_burden')])

> **Decision: KEEP**  
> **Reason:** Clear right-shift for defaulters, meaningful correlation with target, and a one-sentence interview explanation: "What share of monthly income does this payment consume?"

### 2.2 `income_to_loan_ratio`

**Formula:** `annual_inc / loan_amnt`

**Business Rationale:** How large is this loan relative to the borrower's annual income? This is distinct from `monthly_payment_burden` — it ignores interest rate and term, and answers a *relative exposure* question rather than an *affordability* question.

In [ ]:
plot_distribution(df, 'income_to_loan_ratio', 'Income-to-Loan Ratio by Outcome')
plt.tight_layout()
plt.show()

pd.DataFrame([feature_evidence(df, 'income_to_loan_ratio')])

> **Decision: KEEP**  
> **Reason:** Distribution separation is visible and the business question ("how big is this loan relative to what they earn?") is different from the payment-burden question. Redundancy with `monthly_payment_burden` is checked formally in Section 3 — but per Refinement 1, business meaning can override a high correlation if both contribute signal.

### 2.3 `credit_stress_score`

**Formula:** `dti * (revol_util / 100)`

**Business Rationale:** Combines two independent pressure signals — existing debt burden (DTI) and how close the borrower is to maxing their credit lines (utilization) — into a single "compounded stress" measure. A borrower high on both dimensions is in a meaningfully worse position than a borrower high on only one.

In [ ]:
plot_distribution(df, 'credit_stress_score', 'Credit Stress Score by Outcome')
plt.tight_layout()
plt.show()

comparison = pd.DataFrame([
    feature_evidence(df, 'dti'),
    feature_evidence(df, 'revol_util'),
    feature_evidence(df, 'credit_stress_score'),
])
comparison

> **Decision: KEEP**  
> **Reason:** Correlation with target is comparable to its components, and per Refinement 4 this feature is retained on interpretability grounds as long as performance impact is neutral — its formal importance ranking is checked against `dti` and `revol_util` in Section 4. This is the single most "analyst-thinking" feature in the project: it demonstrates the ability to combine two underwriting signals into one interpretable stress measure.

### 2.4 `delinquency_flag`

**Formula:** `1 if delinq_2yrs > 0 else 0`

**Business Rationale:** Past payment behavior is the strongest behavioral predictor of future default. The binary flag is cleaner than the raw count — "any delinquency vs none" matters more than "two delinquencies vs three".

In [ ]:
delinq_summary = (
    df.groupby('delinquency_flag')['target']
    .agg(default_rate=lambda x: x.mean() * 100, count='count')
)
delinq_summary.index = ['No Delinquency', 'Delinquent (2yr)']
print(delinq_summary)

pd.DataFrame([
    feature_evidence(df, 'delinq_2yrs'),
    feature_evidence(df, 'delinquency_flag'),
])

> **Decision: KEEP**  
> **Reason:** Confirmed in Phase B — roughly doubles default risk. The binary flag carries essentially the same correlation as the raw count while being far easier to explain ("has this borrower missed a payment recently, yes or no").

### 2.5 `pub_rec_flag`

**Formula:** `1 if pub_rec > 0 else 0`

**Business Rationale:** Any public derogatory record (bankruptcy, judgment, tax lien) signals a history of severe financial failure. The binary presence matters more than the count.

In [ ]:
pub_rec_summary = (
    df.groupby('pub_rec_flag')['target']
    .agg(default_rate=lambda x: x.mean() * 100, count='count')
)
pub_rec_summary.index = ['No Public Record', 'Has Public Record']
print(pub_rec_summary)

pd.DataFrame([
    feature_evidence(df, 'pub_rec'),
    feature_evidence(df, 'pub_rec_flag'),
])

> **Decision: KEEP**  
> **Reason:** Clear default rate gap, simple binary, easy to explain. Confirmed in Phase B EDA.

### 2.6 `fico_score`

**Formula:** `(fico_range_low + fico_range_high) / 2`

**Business Rationale:** This is a data cleaning step, not a discretionary engineered feature — LendingClub reports a band, and the midpoint is the standard representative value used in underwriting.

In [ ]:
plot_distribution(df, 'fico_score', 'FICO Score by Outcome')
plt.tight_layout()
plt.show()

pd.DataFrame([feature_evidence(df, 'fico_score')])

> **Decision: KEEP (mandatory)**  
> **Reason:** Not optional — this is how the FICO band becomes usable. Strong correlation with target as expected.

### 2.7 `high_utilization_flag` — Threshold Validation

**Formula:** `1 if revol_util > 75 else 0`

**Business Rationale:** Earns its place only if there is a genuine *threshold effect* — i.e., the default rate jumps sharply above 75% rather than increasing smoothly. If the relationship with `revol_util` is roughly linear, the raw variable already captures everything and the flag is redundant.

In [ ]:
# Fine-grained utilization buckets to check for a threshold effect
df['_util_fine'] = pd.cut(
    df['revol_util'],
    bins=list(range(0, 101, 10)),
    labels=[f'{i}-{i+10}%' for i in range(0, 100, 10)]
)

fine_stats = (
    df.groupby('_util_fine')['target']
    .agg(default_rate=lambda x: x.mean() * 100, count='count')
    .query('count >= 100')
)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(fine_stats.index.astype(str), fine_stats['default_rate'],
        marker='o', color=COLOR_DEFAULT, linewidth=2)
ax.axvline(x=7.5, color=COLOR_NEUTRAL, linestyle='--', label='75% threshold')
ax.set_ylabel('Default Rate (%)')
ax.set_xlabel('Revolving Utilization (10% bands)')
ax.set_title('Default Rate vs Utilization — Checking for Threshold Effect')
ax.legend()
plt.tight_layout()
plt.show()

df.drop(columns=['_util_fine'], inplace=True)
print(fine_stats.to_string())

> **Decision: CONDITIONAL — confirmed KEEP**  
> **Reason:** The default-rate curve rises with utilization but does not show a sharp discontinuity — the relationship is closer to gradual than a step function. However, `high_utilization_flag` is retained for two reasons: (1) it directly mirrors the FICO scoring industry convention of a 75% utilization warning level, which is a well-known business rule independent of this dataset's exact curve shape, and (2) it gives the Streamlit pricing engine a simple, explainable rule ("flag if utilization exceeds 75%") without requiring a continuous threshold lookup. Final importance ranking is checked in Section 4 — if it ranks negligibly, it will be reconsidered.

### 2.8 `employment_stability_flag` — Threshold Validation

**Formula:** `1 if emp_length >= 5 else 0`

**Business Rationale:** 5+ years at an employer is a standard underwriting proxy for income stability. Earns its place only if the default-rate curve across employment lengths shows a real inflection near 5 years — not just a smooth, weak decline.

In [ ]:
emp_stats = (
    df.groupby('emp_length')['target']
    .agg(default_rate=lambda x: x.mean() * 100, count='count')
    .query('count >= 100')
    .sort_index()
)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(emp_stats.index.astype(str), emp_stats['default_rate'],
        marker='o', color=COLOR_DEFAULT, linewidth=2)
ax.axvline(x=5, color=COLOR_NEUTRAL, linestyle='--', label='5-year threshold')
ax.set_ylabel('Default Rate (%)')
ax.set_xlabel('Employment Length (years)')
ax.set_title('Default Rate vs Employment Length')
ax.legend()
plt.tight_layout()
plt.show()

spread = emp_stats['default_rate'].max() - emp_stats['default_rate'].min()
print(f'Default rate range across employment lengths: {spread:.2f} percentage points')
print(emp_stats.to_string())

> **Decision: DROP**
> **Reason:** The default rate curve across `emp_length` is essentially flat - there is no
> meaningful inflection point at the 5-year mark (or any other value) that would justify a
> binary cutoff. Since `emp_length` itself is retained in `FINAL_MODEL_FEATURES`, this flag
> would only duplicate information already present in a more granular form. Confirmed as a
> final decision (not conditional).

### 2.9 `loan_term_flag` (New Candidate)

**Formula:** `1 if term == 60 else 0`

**Business Rationale:** 60-month loans default at a materially higher rate than 36-month loans. Term only takes two values, so a binary flag is simply a cleaner representation of the same information — easy to explain: "is this a long-term loan?"

In [ ]:
df['loan_term_flag'] = (df['term'] == 60).astype(int)

term_summary = (
    df.groupby('loan_term_flag')['target']
    .agg(default_rate=lambda x: x.mean() * 100, count='count')
)
term_summary.index = ['36-month', '60-month']
print(term_summary)

> **Decision: KEEP**  
> **Reason:** Large, clean default rate gap between 36- and 60-month loans. Since `term` is binary in this dataset, `loan_term_flag` is a 1:1 recoding that improves readability without losing information — it replaces raw `term` in the model feature list.

### 2.10 `annual_income_log` (Modeling Transformation Only)

**Formula:** `log1p(annual_inc)`

**Status:** Per Refinement 3, this is a **modeling transformation for Logistic Regression only** — not a business feature. It will not appear in the Streamlit app, the README risk-driver discussion, or any business-facing output. `annual_inc` remains the feature business users see.

In [ ]:
from scipy.stats import skew

income_skew = skew(df['annual_inc'].dropna())
income_log_skew = skew(np.log1p(df['annual_inc'].dropna()))

print(f'Skewness of annual_inc:          {income_skew:.2f}')
print(f'Skewness of log1p(annual_inc):   {income_log_skew:.2f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(df['annual_inc'].clip(upper=df['annual_inc'].quantile(0.99)),
              bins=50, color=COLOR_NEUTRAL)
axes[0].set_title(f'annual_inc (skew={income_skew:.2f})')

axes[1].hist(np.log1p(df['annual_inc']), bins=50, color=COLOR_NEUTRAL)
axes[1].set_title(f'log1p(annual_inc) (skew={income_log_skew:.2f})')

plt.tight_layout()
plt.show()

> **Decision: KEEP — Logistic Regression pipeline only**  
> **Reason:** Skewness drops substantially after log transformation, which helps a linear model. CatBoost, being tree-based, is robust to skew and will use raw `annual_inc` directly. This transformation is implemented inside the Logistic Regression preprocessing step in `03_modeling.ipynb`, not as a standalone column in `FINAL_MODEL_FEATURES`.

### 2.11 `credit_lines_per_income` (New Candidate)

**Formula:** `open_acc / annual_inc`

**Business Rationale (proposed):** Borrowers with many open credit lines relative to income may be over-extended.

In [ ]:
df['credit_lines_per_income'] = df['open_acc'] / df['annual_inc'].replace(0, np.nan)

comparison = pd.DataFrame([
    feature_evidence(df, 'open_acc'),
    feature_evidence(df, 'revol_util'),
    feature_evidence(df, 'credit_lines_per_income'),
])
comparison

> **Decision: DROP**  
> **Reason:** `open_acc` (number of accounts) does not capture *how much* of those credit lines is used — `revol_util` already measures credit stress directly and more meaningfully. `credit_lines_per_income` conflates credit *access* with credit *stress*, has weaker correlation with the target than `revol_util`, and is harder to explain in an interview ("why does the number of accounts divided by income matter, specifically?"). Removed from the candidate list.

---
## Section 3 — Redundancy Check: "Do We Really Need This Feature?"

A focused correlation table — not a full heatmap — between candidate features that might tell similar stories.

In [ ]:
redundancy_pairs = [
    'monthly_payment_burden', 'income_to_loan_ratio',
    'dti', 'revol_util', 'credit_stress_score',
    'fico_score', 'int_rate',
    'delinquency_flag', 'pub_rec_flag',
]

corr_table = df[redundancy_pairs].corr().round(2)
corr_table

### Redundancy Findings

**`monthly_payment_burden` vs `income_to_loan_ratio`**  
Both relate income to loan size, so some correlation is expected. However they answer different questions — monthly burden incorporates interest rate and term (affordability of *this specific structure*), while income-to-loan ignores both (relative *size of exposure*). Per Refinement 1, business meaning overrides a moderate correlation here. **Both kept.**

**`dti` vs `revol_util` vs `credit_stress_score`**  
`credit_stress_score` is built from the other two, so correlation with each is structurally expected. The question is whether it adds anything *beyond* its components — addressed directly in Section 4 via Logistic Regression coefficients.

**`fico_score` vs `int_rate`**  
These are likely to show meaningful correlation, since LendingClub itself prices loans partly based on credit score. Both are kept: `fico_score` is the underlying credit quality signal, while `int_rate` reflects LendingClub's own pricing decision — useful as a benchmark for our own pricing engine in Phase 8.

**`delinquency_flag` vs `pub_rec_flag`**  
Conceptually distinct — one is payment history, the other is public legal/financial record. Low correlation expected; both kept on separate business grounds.

---
## Section 4 — Feature Importance Preview (Logistic Regression)

We train a quick Logistic Regression on the candidate feature set to rank-order features. This uses the **same model family as our Phase D baseline**, so the conclusions here remain consistent with the eventual model — we are not introducing a separate model that gets discarded afterward.

This is a **preview**, not the final model. No hyperparameter tuning, no train/test evaluation reporting — purely for ranking.

In [ ]:
# Candidate feature set for the importance preview.
# Categorical columns are one-hot encoded for Logistic Regression here only —
# this encoding is local to this preview and re-done properly in 03_modeling.ipynb.
preview_numeric = [
    'loan_amnt', 'int_rate', 'annual_inc', 'dti', 'revol_util', 'fico_score',
    'emp_length', 'sub_grade', 'delinq_2yrs', 'pub_rec',
    'monthly_payment_burden', 'income_to_loan_ratio', 'credit_stress_score',
    'delinquency_flag', 'pub_rec_flag', 'high_utilization_flag',
    'employment_stability_flag', 'loan_term_flag',
]
preview_categorical = ['grade', 'purpose', 'home_ownership']

preview_df = df[preview_numeric + preview_categorical + ['target']].dropna()

X = pd.get_dummies(preview_df[preview_numeric + preview_categorical],
                    columns=preview_categorical, drop_first=True)
y = preview_df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardize numeric features — required for meaningful LR coefficient comparison
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

lr_preview = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_preview.fit(X_train_scaled, y_train)

print('Preview model trained.')

In [ ]:
# Coefficient magnitude = importance, since features are standardized
importance = pd.DataFrame({
    'feature': X.columns,
    'coefficient': lr_preview.coef_[0],
    'abs_coefficient': np.abs(lr_preview.coef_[0]),
}).sort_values('abs_coefficient', ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(9, 8))
top_n = importance.head(20)
colors = [COLOR_DEFAULT if c > 0 else COLOR_NONDEFAULT for c in top_n['coefficient']]
ax.barh(top_n['feature'][::-1], top_n['coefficient'][::-1], color=colors[::-1])
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_xlabel('Standardized Coefficient (red = increases default risk)')
ax.set_title('Logistic Regression Feature Importance Preview (Top 20)')
plt.tight_layout()
plt.show()

importance.head(20)

In [ ]:
# Specific check: where do our conditional/engineered features rank?
watch_list = [
    'monthly_payment_burden', 'income_to_loan_ratio', 'credit_stress_score',
    'dti', 'revol_util',
    'high_utilization_flag', 'employment_stability_flag', 'emp_length',
    'delinquency_flag', 'pub_rec_flag', 'loan_term_flag',
]

importance['rank'] = importance.index + 1
watch_results = importance[importance['feature'].isin(watch_list)].sort_values('rank')
print(f'Total features in preview model: {len(importance)}')
watch_results

### Resolving the Conditional Features

**`credit_stress_score`**  
Compare its rank to `dti` and `revol_util` individually (printed above). If `credit_stress_score` ranks reasonably (not at the bottom of the table) and its coefficient sign is consistent (positive — higher stress, higher risk), it is retained per Refinement 4 on interpretability grounds, even if `dti` and `revol_util` individually rank higher. The combination still adds an analyst-style narrative the standalone variables don't provide.

**`high_utilization_flag`**  
If its coefficient is near zero and its rank is in the bottom quartile, it contributes little once `revol_util` is already present — but it is retained anyway for the pricing engine's rule-based logic (Phase 8), where a simple yes/no "high utilization" rule is more usable than a continuous threshold lookup. Its presence in the model is low-cost (one extra column) and its business utility in the dashboard outweighs a marginal statistical contribution.

**`employment_stability_flag`**  
If this ranks at or near the bottom with a coefficient close to zero, **it is dropped**. Section 2.8 already showed a flat default-rate curve — without a meaningful coefficient here as well, this feature fails two of the three retention criteria (weak business signal AND weak measurable signal). Raw `emp_length` remains in the model and carries whatever weak signal exists.

---
## Section 5 — Feature Review Board

This table documents the underwriting logic behind every feature decision in this notebook. It doubles as README content — a recruiter can read this table alone and understand the entire feature engineering philosophy.

In [ ]:
review_board = pd.DataFrame([
    {'Feature': 'sub_grade', 'Business Logic': 'Granular LendingClub risk grade (A1-G5), ordinal', 'Signal Strength': 'Strong', 'Redundancy Risk': 'High vs grade - grade dropped as redundant refinement', 'Decision': 'KEEP'},
    {'Feature': 'fico_score', 'Business Logic': 'Midpoint of reported FICO band - bureau credit score', 'Signal Strength': 'Strong', 'Redundancy Risk': 'Low', 'Decision': 'KEEP (mandatory)'},
    {'Feature': 'dti', 'Business Logic': 'Existing debt burden relative to income', 'Signal Strength': 'Strong', 'Redundancy Risk': 'Low - distinct from revol_util (debt vs utilization)', 'Decision': 'KEEP'},
    {'Feature': 'revol_util', 'Business Logic': 'Revolving credit utilization - proximity to credit limits', 'Signal Strength': 'Strong', 'Redundancy Risk': 'Low - distinct from dti', 'Decision': 'KEEP'},
    {'Feature': 'int_rate', 'Business Logic': 'Market pricing of risk by LendingClub', 'Signal Strength': 'Strong', 'Redundancy Risk': 'Moderate vs sub_grade - kept as distinct market-pricing view, useful for Phase 8 pricing benchmark', 'Decision': 'KEEP'},
    {'Feature': 'monthly_payment_burden', 'Business Logic': 'Installment as % of monthly income - affordability of this loan structure', 'Signal Strength': 'Strong', 'Redundancy Risk': 'Moderate vs income_to_loan_ratio - different question (affordability vs exposure size), both kept', 'Decision': 'KEEP'},
    {'Feature': 'delinquency_flag', 'Business Logic': 'Any delinquency in past 2 years - behavioral history', 'Signal Strength': 'Strong', 'Redundancy Risk': 'Low - cleaner binary form of delinq_2yrs (delinq_2yrs dropped)', 'Decision': 'KEEP'},
    {'Feature': 'loan_term_flag', 'Business Logic': '60-month vs 36-month loan structure', 'Signal Strength': 'Strong', 'Redundancy Risk': 'None - 1:1 recoding of term', 'Decision': 'KEEP'},
    {'Feature': 'income_to_loan_ratio', 'Business Logic': 'Loan size relative to annual income - exposure sizing', 'Signal Strength': 'Moderate', 'Redundancy Risk': 'Moderate vs monthly_payment_burden - different question (exposure vs affordability), both kept', 'Decision': 'KEEP'},
    {'Feature': 'loan_amnt', 'Business Logic': 'Absolute size of credit exposure', 'Signal Strength': 'Moderate', 'Redundancy Risk': 'Moderate vs installment - installment dropped as redundant with monthly_payment_burden; loan_amnt kept for expected-loss calc', 'Decision': 'KEEP'},
    {'Feature': 'annual_inc', 'Business Logic': 'Borrower annual income - primary repayment resource', 'Signal Strength': 'Moderate', 'Redundancy Risk': 'Low - absolute capacity distinct from ratio features', 'Decision': 'KEEP'},
    {'Feature': 'pub_rec_flag', 'Business Logic': 'Any public derogatory record (bankruptcy/lien/judgment)', 'Signal Strength': 'Moderate', 'Redundancy Risk': 'Low - cleaner binary form of pub_rec and pub_rec_bankruptcies (both dropped)', 'Decision': 'KEEP'},
    {'Feature': 'purpose', 'Business Logic': 'Stated reason for the loan - behavioral/intent signal', 'Signal Strength': 'Weak-Moderate', 'Redundancy Risk': 'Low - categorical with no numeric overlap', 'Decision': 'KEEP'},
    {'Feature': 'home_ownership', 'Business Logic': 'Rent/Own/Mortgage - stability and collateral proxy', 'Signal Strength': 'Weak', 'Redundancy Risk': 'Low', 'Decision': 'KEEP'},
    {'Feature': 'verification_status', 'Business Logic': 'Whether income was verified by LendingClub', 'Signal Strength': 'Weak', 'Redundancy Risk': 'Low - retained for documented selection-bias finding (verified borrowers sometimes show higher default rates)', 'Decision': 'KEEP'},
    {'Feature': 'emp_length', 'Business Logic': 'Years at current employer - income stability proxy', 'Signal Strength': 'Weak', 'Redundancy Risk': 'High vs employment_stability_flag - flag dropped, raw value retained directly', 'Decision': 'KEEP'},
    {'Feature': 'credit_history_years', 'Business Logic': 'Length of credit history in years', 'Signal Strength': 'Weak', 'Redundancy Risk': 'Low', 'Decision': 'KEEP'},
    {'Feature': 'high_utilization_flag', 'Business Logic': 'Revolving utilization above 75% - FICO-aligned threshold rule', 'Signal Strength': 'Weak-Moderate', 'Redundancy Risk': 'High vs revol_util - kept for pricing-engine rule logic, not statistical independence', 'Decision': 'KEEP'},
    {'Feature': 'grade', 'Business Logic': 'LendingClub coarse risk grade (A-G)', 'Signal Strength': 'Strong', 'Redundancy Risk': 'Very high vs sub_grade - sub_grade is a strict refinement', 'Decision': 'DROP - redundant with sub_grade'},
    {'Feature': 'installment', 'Business Logic': 'Monthly repayment amount', 'Signal Strength': 'Moderate', 'Redundancy Risk': 'High vs monthly_payment_burden - burden already captures affordability', 'Decision': 'DROP - overlaps with monthly_payment_burden'},
    {'Feature': 'delinq_2yrs', 'Business Logic': 'Count of delinquencies in past 2 years', 'Signal Strength': 'Weak', 'Redundancy Risk': 'High vs delinquency_flag - flag is derived directly from this column', 'Decision': 'DROP - redundant with delinquency_flag'},
    {'Feature': 'pub_rec', 'Business Logic': 'Count of public derogatory records', 'Signal Strength': 'Weak', 'Redundancy Risk': 'High vs pub_rec_flag - flag is derived directly from this column', 'Decision': 'DROP - redundant with pub_rec_flag'},
    {'Feature': 'pub_rec_bankruptcies', 'Business Logic': 'Count of public bankruptcy records', 'Signal Strength': 'Weak', 'Redundancy Risk': 'High vs pub_rec_flag - same underlying signal', 'Decision': 'DROP - redundant with pub_rec_flag'},
    {'Feature': 'revol_bal', 'Business Logic': 'Outstanding revolving balance (absolute)', 'Signal Strength': 'Weak', 'Redundancy Risk': 'High vs revol_util - utilization is the normalized and more interpretable version', 'Decision': 'DROP - redundant with revol_util'},
    {'Feature': 'open_acc', 'Business Logic': 'Number of open credit lines', 'Signal Strength': 'Weak', 'Redundancy Risk': 'Low signal - not tied to any EDA finding', 'Decision': 'DROP - weak undefended signal'},
    {'Feature': 'total_acc', 'Business Logic': 'Total credit lines ever opened', 'Signal Strength': 'Weak', 'Redundancy Risk': 'Low signal - not tied to any EDA finding', 'Decision': 'DROP - weak undefended signal'},
    {'Feature': 'mort_acc', 'Business Logic': 'Number of mortgage accounts', 'Signal Strength': 'Weak', 'Redundancy Risk': 'Low signal - not tied to any EDA finding', 'Decision': 'DROP - weak undefended signal'},
    {'Feature': 'inq_last_6mths', 'Business Logic': 'Credit inquiries in past 6 months', 'Signal Strength': 'Weak', 'Redundancy Risk': 'Low signal - not tied to any EDA finding', 'Decision': 'DROP - weak undefended signal'},
    {'Feature': 'employment_stability_flag', 'Business Logic': '5+ years employment tenure (binary)', 'Signal Strength': 'Weak', 'Redundancy Risk': 'High vs emp_length - flat default-rate curve in EDA showed no inflection at 5 years', 'Decision': 'DROP - emp_length retained directly with less duplication'},
    {'Feature': 'credit_lines_per_income', 'Business Logic': 'Open credit lines relative to income (proposed)', 'Signal Strength': 'Weak', 'Redundancy Risk': 'Conflates credit access with credit stress - revol_util superior', 'Decision': 'DROP - weak business case'},
    {'Feature': 'annual_income_log', 'Business Logic': 'Log transform of annual_inc for linear model', 'Signal Strength': 'N/A - transformation not a risk factor', 'Redundancy Risk': 'Direct transform of annual_inc', 'Decision': 'KEEP - Logistic Regression pipeline only, not in FINAL_MODEL_FEATURES'},
    {'Feature': 'utilization_bucket', 'Business Logic': 'Low/Medium/High utilization band for dashboard', 'Signal Strength': 'N/A - display only', 'Redundancy Risk': 'Direct bucketing of revol_util', 'Decision': 'DISPLAY ONLY - EDA / Streamlit'},
    {'Feature': 'loan_to_income_bucket', 'Business Logic': 'Exposure tier for dashboard display', 'Signal Strength': 'N/A - display only', 'Redundancy Risk': 'Direct bucketing of income_to_loan_ratio', 'Decision': 'DISPLAY ONLY - EDA / Streamlit'},
])

review_board


> **Note:** Both previously-conditional features have now been finalized. `high_utilization_flag`
> is **KEPT** at the 75% threshold (FICO-aligned industry convention, confirmed in Section 2.7).
> `employment_stability_flag` is **DROPPED** - the default rate curve across `emp_length` showed
> no meaningful inflection point, so `emp_length` is retained directly instead (Section 2.8).


In [ ]:
# Both previously-conditional features are now finalized (see note above):
#   - high_utilization_flag -> KEEP (75% threshold, FICO-aligned)
#   - employment_stability_flag -> DROP (emp_length retained directly)
#
# These decisions are already reflected in the review_board table above and
# in src/features.py's FINAL_MODEL_FEATURES. No further resolution needed.
print("Conditional features finalized: high_utilization_flag=KEEP, employment_stability_flag=DROP")


---
## Section 6 — Locking FINAL_MODEL_FEATURES

Based on the evidence and decisions above, the final feature list is assembled and written back to `src/features.py`. From this point forward, `03_modeling.ipynb`, `04_calibration.ipynb`, `05_business_analytics.ipynb`, and the Streamlit app all import this single list.

In [ ]:
# Final feature lists, locked per the feature governance review.
# 12 raw + 7 engineered = 19 features total (within the 18-20 target).

RAW_MODEL_FEATURES = [
    'loan_amnt', 'int_rate', 'sub_grade', 'fico_score', 'annual_inc', 'dti',
    'revol_util', 'emp_length', 'credit_history_years', 'home_ownership',
    'purpose', 'verification_status',
]

ENGINEERED_MODEL_FEATURES = [
    'monthly_payment_burden', 'income_to_loan_ratio', 'credit_stress_score',
    'delinquency_flag', 'pub_rec_flag', 'loan_term_flag', 'high_utilization_flag',
]

FINAL_MODEL_FEATURES = RAW_MODEL_FEATURES + ENGINEERED_MODEL_FEATURES

print(f"Raw features: {len(RAW_MODEL_FEATURES)}")
print(f"Engineered features: {len(ENGINEERED_MODEL_FEATURES)}")
print(f"FINAL_MODEL_FEATURES total: {len(FINAL_MODEL_FEATURES)}")
FINAL_MODEL_FEATURES


In [ ]:
# FINAL_MODEL_FEATURES has already been written to src/features.py as part of
# the feature governance review (with full rationale comments documenting why
# each raw column was kept or dropped). Verify the lists match exactly.

import importlib
import features
importlib.reload(features)

assert features.FINAL_MODEL_FEATURES == FINAL_MODEL_FEATURES, \
    "Mismatch between notebook FINAL_MODEL_FEATURES and src/features.py"

print("src/features.py FINAL_MODEL_FEATURES matches notebook decision (19 features).")


---
## Section 7 — Save Outputs

In [ ]:
# Build outputs/feature_summary.csv (Feature, Type, Business Meaning, Selected, Reason)
# Mirrors the review_board decisions above, scoped to all 19 kept features plus
# the dropped/rejected features and modeling-only/display-only entries.

business_meaning = {
    'loan_amnt': 'Size of credit exposure',
    'int_rate': 'LendingClub-assigned interest rate (market pricing of risk)',
    'sub_grade': 'Granular LendingClub risk grade (A1-G5) ordinally encoded',
    'fico_score': 'Credit bureau score (midpoint of reported band)',
    'annual_inc': 'Borrower annual income (repayment capacity)',
    'dti': 'Debt-to-income ratio (existing debt burden)',
    'revol_util': 'Revolving credit utilization (%)',
    'emp_length': 'Years at current employer (0-10)',
    'credit_history_years': 'Length of credit history in years',
    'home_ownership': 'Rent / Own / Mortgage status',
    'purpose': 'Stated reason for the loan',
    'verification_status': 'Whether income was verified by LendingClub',
    'monthly_payment_burden': 'Installment as % of monthly income (affordability)',
    'income_to_loan_ratio': 'Loan size relative to annual income (exposure sizing)',
    'credit_stress_score': 'Combined DTI x utilization stress measure',
    'delinquency_flag': 'Any delinquency in past 2 years (binary)',
    'pub_rec_flag': 'Any public derogatory record (binary)',
    'loan_term_flag': '60-month loan vs 36-month loan (binary)',
    'high_utilization_flag': 'Revolving utilization above 75% (binary)',
    'grade': 'LendingClub coarse risk grade (A-G)',
    'installment': 'Monthly repayment amount',
    'delinq_2yrs': 'Count of delinquencies in past 2 years',
    'pub_rec': 'Count of public derogatory records',
    'pub_rec_bankruptcies': 'Count of public bankruptcy records',
    'revol_bal': 'Outstanding revolving balance',
    'open_acc': 'Number of open credit lines',
    'total_acc': 'Total credit lines ever opened',
    'mort_acc': 'Number of mortgage accounts',
    'inq_last_6mths': 'Credit inquiries in past 6 months',
    'employment_stability_flag': '5+ years employment tenure (binary)',
    'credit_lines_per_income': 'Open credit lines relative to income',
    'annual_income_log': 'Log transform of annual_inc - reduces skew for linear model',
    'utilization_bucket': 'Low/Medium/High utilization band for dashboard',
    'loan_to_income_bucket': 'Exposure tier for dashboard display',
}

feature_type = {f: 'Raw' for f in RAW_MODEL_FEATURES}
feature_type.update({f: 'Engineered' for f in ENGINEERED_MODEL_FEATURES})
for f in ['grade', 'installment', 'delinq_2yrs', 'pub_rec', 'pub_rec_bankruptcies',
          'revol_bal', 'open_acc', 'total_acc', 'mort_acc', 'inq_last_6mths']:
    feature_type[f] = 'Raw (rejected)'
feature_type['employment_stability_flag'] = 'Engineered (rejected)'
feature_type['credit_lines_per_income'] = 'Engineered (rejected candidate)'
feature_type['annual_income_log'] = 'Modeling transformation (LR only)'
feature_type['utilization_bucket'] = 'Display only'
feature_type['loan_to_income_bucket'] = 'Display only'

reason_map = dict(zip(review_board['Feature'], review_board['Decision']))

summary_rows = []
for feat, meaning in business_meaning.items():
    selected = 'Yes' if feat in FINAL_MODEL_FEATURES else 'No'
    summary_rows.append({
        'Feature': feat,
        'Type': feature_type[feat],
        'Business Meaning': meaning,
        'Selected': selected,
        'Reason': reason_map.get(feat, ''),
    })

feature_summary_final = pd.DataFrame(summary_rows)

review_board.to_csv('../outputs/feature_review_board.csv', index=False)
feature_summary_final.to_csv('../outputs/feature_summary.csv', index=False)

print(f"Saved feature_review_board.csv ({len(review_board)} rows)")
print(f"Saved feature_summary.csv ({len(feature_summary_final)} rows)")
feature_summary_final


---
## Section 8 - Summary

**Final feature count: 19** (within the 18-20 target set by the governance review)
- **12 raw features:** `loan_amnt`, `int_rate`, `sub_grade`, `fico_score`, `annual_inc`, `dti`,
  `revol_util`, `emp_length`, `credit_history_years`, `home_ownership`, `purpose`, `verification_status`
- **7 engineered features:** `monthly_payment_burden`, `income_to_loan_ratio`, `credit_stress_score`,
  `delinquency_flag`, `pub_rec_flag`, `loan_term_flag`, `high_utilization_flag`

### Governance overrides applied in this final pass
- **`int_rate` - KEPT.** Represents the market's pricing of risk, conceptually distinct from
  `fico_score` (bureau view) and `sub_grade` (LendingClub's internal view). Strengthens the
  pricing narrative for Phase 8.
- **`loan_amnt` - KEPT.** Directly interpretable exposure size, used later in expected-loss
  (PD x LGD x EAD) calculations.
- **`installment` - DROPPED.** `monthly_payment_burden` already captures affordability relative
  to income; keeping both would be redundant.
- **`employment_stability_flag` - DROPPED.** The default-rate curve across `emp_length` showed
  no meaningful inflection at the 5-year (or any) cutoff. `emp_length` is retained directly.

### What was dropped and why
- `grade` - redundant with `sub_grade` (strict refinement)
- `delinq_2yrs`, `pub_rec`, `pub_rec_bankruptcies`, `revol_bal` - redundant with their derived
  binary flags or `revol_util`
- `open_acc`, `total_acc`, `mort_acc`, `inq_last_6mths` - weak, undefended signal
- `credit_lines_per_income` - conflates credit access with credit stress; `revol_util` is superior
- `annual_income_log` - modeling-only transform for the Logistic Regression pipeline, not a
  business risk factor, so excluded from `FINAL_MODEL_FEATURES`

### Single source of truth
`FINAL_MODEL_FEATURES` is locked in `src/features.py` (with full rationale comments) and is
imported unchanged by `03_modeling.ipynb`, `05_business_analytics.ipynb`, and the Streamlit app.
Supporting evidence is saved to `outputs/feature_review_board.csv` and `outputs/feature_summary.csv`.

### Interview test
*"Why is `int_rate` in the model if `sub_grade` already encodes risk?"* - Because `int_rate`
reflects LendingClub's realized market pricing decision, while `sub_grade` is the underlying
risk bucket. Together they let us compare the bank's internal risk view against the price the
market actually charged - a key input to the risk-based pricing engine in Phase 8.
